# P01 - Baseline global con `PRI_jet_num` categorica

Este notebook toma el baseline `P00` y cambia una sola decision metodologica importante: `PRI_jet_num` deja de tratarse como cantidad ordinal y pasa a codificarse como variable categorica estructural. Se generan siempre las cuatro columnas fijas `PRI_jet_num_0`, `PRI_jet_num_1`, `PRI_jet_num_2` y `PRI_jet_num_3`.

El resto del flujo se mantiene igual: indicadores de faltantes aprendidos en train, `log1p` en columnas positivas, imputacion por mediana y `StandardScaler` para las variables continuas. Las columnas one-hot y los indicadores no se escalan.

Salidas principales en `data/preprocesor/P01_jet_onehot_standard/`.



## Imports y configuracion


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_work_dir(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        direct = candidate / "preprocessing_lib.py"
        nested = candidate / "50. preprocesamiento" / "preprocessing_lib.py"
        if direct.exists():
            return candidate
        if nested.exists():
            return candidate / "50. preprocesamiento"
    raise FileNotFoundError("No se pudo ubicar 50. preprocesamiento/preprocessing_lib.py")


work_dir = find_work_dir()
if str(work_dir) not in sys.path:
    sys.path.insert(0, str(work_dir))

from preprocessing_lib import (
    build_common_data,
    output_root,
    save_common_artifacts,
    save_global_pipeline_artifacts,
    save_p03_artifacts,
    split_summary,
    transform_global_pipeline,
    transform_p03,
)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)



## Ajuste y transformacion


In [2]:
pipeline_id = "P01"
common = build_common_data()
result = transform_global_pipeline(pipeline_id, common)
output_dir = output_root() / "P01_jet_onehot_standard"

print(f"Directorio de salida: {output_dir}")
print("Resumen del split:")
display(split_summary(common).round(6))

print("Shapes procesados:")
shape_df = pd.DataFrame([
    {"split": "train", "rows": result["X_train_processed"].shape[0], "features": result["X_train_processed"].shape[1], "nan_count": int(result["X_train_processed"].isna().sum().sum())},
    {"split": "valid", "rows": result["X_valid_processed"].shape[0], "features": result["X_valid_processed"].shape[1], "nan_count": int(result["X_valid_processed"].isna().sum().sum())},
    {"split": "test", "rows": result["X_test_processed"].shape[0], "features": result["X_test_processed"].shape[1], "nan_count": int(result["X_test_processed"].isna().sum().sum())},
])
display(shape_df)

print("Primeras features finales:")
display(pd.DataFrame({"feature": result["metadata"]["final_feature_order"]}).head(25))

print("Columnas con missing flags:")
display(pd.DataFrame({"feature": result["metadata"]["missing_indicator_cols"]}))


Directorio de salida: /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P01_jet_onehot_standard
Resumen del split:


,split,rows,class_b,class_s,weight_sum
0,train,225000,147900.0,77100.0,370822.739836
1,valid,25000,16433.0,8567.0,40869.096094
2,test,550000,NaN,NaN,NaN


Shapes procesados:


,split,rows,features,nan_count
0,train,225000,44,0
1,valid,25000,44,0
2,test,550000,44,0


Primeras features finales:


,feature
0,DER_mass_MMC
1,DER_mass_transverse_met_lep
2,DER_mass_vis
3,DER_pt_h
4,DER_deltaeta_jet_jet
5,DER_mass_jet_jet
6,DER_prodeta_jet_jet
7,DER_deltar_tau_lep
8,DER_pt_tot
9,DER_sum_pt


Columnas con missing flags:


,feature
0,DER_mass_MMC
1,DER_deltaeta_jet_jet
2,DER_mass_jet_jet
3,DER_prodeta_jet_jet
4,DER_lep_eta_centrality
5,PRI_jet_leading_pt
6,PRI_jet_leading_eta
7,PRI_jet_leading_phi
8,PRI_jet_subleading_pt
9,PRI_jet_subleading_eta


## Guardado de artefactos


In [3]:
saved_paths = save_global_pipeline_artifacts(pipeline_id, common, result, output_dir)
print(f"Artefactos guardados: {len(saved_paths)}")
for path in saved_paths:
    print(f"- {path}")


Artefactos guardados: 16
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P01_jet_onehot_standard/X_train_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P01_jet_onehot_standard/X_valid_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P01_jet_onehot_standard/X_test_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P01_jet_onehot_standard/y_train.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P01_jet_onehot_standard/y_valid.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P01_jet_onehot_standard/train_weight.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P01_jet_onehot_standard/valid_weight.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P01_jet_onehot_standard/train_event_id.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P01_jet_onehot_standard/valid_event_id.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P01_jet_onehot_standard/test_event_id.csv
- /home/santig14